# 第 9 章　FX Trading with FXCM

> *Financial institutions like to call what they do trading. Let's be honest. It's not trading; it's betting.* —— Graydon Carter


## 0. 引言

本章介绍 FXCM Group, LLC（以下简称 FXCM）的交易平台：REST 与流式 API，以及 Python 封装 `fxcmpy`。与第 8 章 Oanda 类似，该平台面向零售与机构交易者，可部署自动化算法交易策略。产品侧重货币对，以及股指、商品等差价合约（CFD）。

选择交易平台时，可沿用第 8 章的维度：

- Instruments（交易品种）：外汇、股指 CFD、商品 CFD 等。
- Strategies（策略）：可杠杆做多/做空；支持市价入场、止损、止盈等。
- Costs（成本）：买卖价差之外，通常另有按笔固定费用；具体以账户定价模型为准。
- Technology（技术）：现代 REST API，可用 `fxcmpy` 访问；另有桌面/平板/手机交易端。
- Jurisdiction（司法辖区）：业务覆盖多国，产品可用性受当地法规约束。

本 notebook 对照原书 Hilpisch《Python for Algorithmic Trading》 第 9 章，体例对齐 `CFD_Trading_with_Oanda.ipynb`。目标是理清 FXCM API 流水线：连接 → 历史/流式数据 → 下单 → 账户查询。

### 0.1 大陆学习路径说明

FXCM 是境外外汇/CFD 经纪通道，与国内期货程序化接入无直接对应关系。大陆对个人杠杆外汇/CFD 的合规约束显著，不宜作为默认可运行教学路径。

API 形态与 Oanda 相同；若关心 “会话 → 行情 → 信号 → 下单 → 账户” 的可复现实现，请以天勤笔记 `天勤TqSdk期货算法交易.ipynb` 为主。

| 维度 | Oanda | FXCM | 天勤 TqSdk |
|---|---|---|---|
| 标的 | 外汇、CFD 等 | 外汇、股指/商品等 CFD | 国内商品/金融期货等 |
| 账户模型 | 多为净持仓（同品种多空轧差） | 演示账户常为 hedge（同品种多空可分仓并存） | 按合约持仓；可用 `TargetPosTask` 设目标手数 |
| 资金账户 | Oanda 经纪账户 | FXCM 经纪账户 | 期货公司账户（`TqAccount`）；模拟用 `TqSim` |
| API 认证 | account id + token | API token（`fxcmpy`） | 快期 `TqAuth` +（实盘）期货公司凭证 |

### 0.2 风险提示（Disclaimer）

外汇/CFD 保证金交易风险很高，亏损可能超过入金。产品面向零售与专业客户；具体保护条款因实体与居住地而异。任何行情评论不构成投资建议。交易前请评估自身财务状况与经验。平台与作者不对直接或间接损失承担责任。


## 1. Getting Started

官方 API 文档：https://fxcm.github.io/rest-api-docs  
`fxcmpy` 文档：http://fxcmpy.tpq.io

安装：


In [ ]:
# !pip install fxcmpy -i https://mirrors.aliyun.com/pypi/simple


开通免费演示账户（部分国家/地区可能不提供 Demo）：按 FXCM 官网流程注册后，在账户内生成唯一 API token。

与第 8 章一样，推荐用配置文件管理凭证。在笔记本同目录创建 `fxcm.cfg`（请勿提交 Git）：

```ini
[FXCM]
log_level = error
log_file = fxcm.log
access_token = YOUR_FXCM_API_TOKEN
```

连接示例：


In [ ]:
import fxcmpy
import datetime as dt
import numpy as np
import pandas as pd
from pylab import mpl, plt

plt.style.use("seaborn-v0_8")
mpl.rcParams["savefig.dpi"] = 300
mpl.rcParams["font.family"] = "serif"
plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei", "SimHei", "PingFang SC", "Noto Sans CJK SC", "Arial Unicode MS"
]
plt.rcParams["axes.unicode_minus"] = False

# 有可用 Demo/实盘 token 时取消注释；国内环境常无法连接，后续单元格附书中示意输出
# api = fxcmpy.fxcmpy(config_file="fxcm.cfg", server="demo")  # ① demo 服务器
# api = fxcmpy.fxcmpy(config_file="fxcm.cfg", server="real")  # ② 实盘服务器（需真实账户）
print("fxcmpy 版本：", getattr(fxcmpy, "__version__", "(未知)"))
print("已 import。连接请取消上方注释并配置 fxcm.cfg。")


`server='demo'` 连接演示服务器；`server='real'` 连接实盘。默认通常为 demo。

连上 API 后，即可获取历史数据、订阅流式行情、下单与查询账户。若本机无法连通，请把下列代码当作书中流水线阅读，可运行复现请转天勤笔记。


## 2. Retrieving Data

除 API 在线拉取外，FXCM 还提供按周打包的历史 tick / candles 文件，经 `fxcmpy` 的 reader 下载解压。适合离线分析；与第 8 章「只走 REST 拉历史」略有不同。

### 2.1 Tick Data

可用货币对列表：


In [1]:
from fxcmpy import fxcmpy_tick_data_reader as tdr

print(tdr.get_available_symbols())
# 书中示意：('AUDCAD', 'AUDCHF', ..., 'EURUSD', ..., 'USDJPY')

ModuleNotFoundError: No module named 'fxcmpy'

按下述方式取一周 tick（数据量可能达数百万行）。`.get_raw_data()` 返回原始索引（字符串）；`.get_data()` 将索引转为 `DatetimeIndex`（转换较耗时）：


In [ ]:
# 需要能访问 FXCM 数据服务器；失败时跳过即可
# start = dt.datetime(2020, 3, 25)
# stop = dt.datetime(2020, 3, 30)
# td = tdr("EURUSD", start, stop)   # ① 下载并解包为 DataFrame
# td.get_raw_data().info()          # ② 原始索引仍为 str
# td.get_data().info()              # ③ DatetimeIndex
# td.get_data().head()

# 书中示意片段（Mid + SMA）
# sub = td.get_data(start="2020-03-25 12:00:00", end="2020-03-25 12:15:00")
# sub["Mid"] = sub.mean(axis=1)                    # ④ bid/ask 中间价
# sub["SMA"] = sub["Mid"].rolling(1000).mean()     # ⑤ 1000 tick SMA
# sub[["Mid", "SMA"]].plot(figsize=(10, 6), lw=1.5)
print("Tick reader 用法已写在注释中；有网络与权限时取消注释运行。")


### 2.2 Candles Data

Candles 为均匀时间间隔上的 OHLC（bid/ask 各一套）。需指定 `period`，例如 `m1`（1 分钟）、`H1`（1 小时）、`D1`（日线）：


In [ ]:
from fxcmpy import fxcmpy_candles_data_reader as cdr

print(cdr.get_available_symbols())

# start = dt.datetime(2020, 4, 1)
# stop = dt.datetime(2020, 5, 1)
# period = "H1"   # ① 周期
# candles = cdr("EURUSD", start, stop, period)
# data = candles.get_data()
# data.info()
# data[data.columns[:4]].tail()   # ② Bid OHLC
# data[data.columns[4:]].tail()   # ③ Ask OHLC

# data["MidClose"] = data[["BidClose", "AskClose"]].mean(axis=1)  # ④ 中间收盘价
# data["SMA1"] = data["MidClose"].rolling(30).mean()
# data["SMA2"] = data["MidClose"].rolling(100).mean()
# data[["MidClose", "SMA1", "SMA2"]].plot(figsize=(10, 6))
print("Candles reader 用法已写在注释中。")


## 3. Working with the API

上一节是预打包文件；本节通过已连接的 `api` 对象拉取历史、订阅流式行情、下单。可交易品种可能比预打包集合更广。

### 3.1 连接与品种列表


In [ ]:
# instruments = api.get_instruments()
# print(instruments)

# 书中示意片段
instruments_demo = [
    "EUR/USD", "USD/JPY", "GBP/USD", "USD/CHF", "EUR/CHF",
    "AUD/USD", "USD/CAD", "NZD/USD", "GER30", "SPX500",
    "USOil", "XAU/USD", "BTC/USD",
]
print("示意品种（前若干）：", instruments_demo)
print("完整列表请在 api 连接成功后调用 api.get_instruments()。")


### 3.2 通过 API 取历史 Candles

`.get_candles()` 的 `period` 可为：`m1`, `m5`, `m15`, `m30`, `H1`, `H2`, `H3`, `H4`, `H6`, `H8`, `D1`, `W1`, `M1`。

可按最近条数，或按起止日期拉取：


In [ ]:
# candles = api.get_candles("USD/JPY", period="D1", number=10)  # ① 最近 10 根日线
# candles[candles.columns[:4]]   # bid 侧
# candles[candles.columns[4:]]   # ask 侧 + tickqty

# start = dt.datetime(2019, 1, 1)
# end = dt.datetime(2020, 6, 1)
# candles = api.get_candles("EUR/GBP", period="D1", start=start, stop=end)  # ② 区间
# candles.info()

# candles = api.get_candles("EUR/USD", period="m1", number=250)  # ③ 最近 250 根 1 分钟线
# candles["askclose"].plot(figsize=(10, 6))

print("注意：REST 返回的 bid/ask 可能随账户定价模型变化。")


### 3.3 Streaming Data

与 Oanda 类似，FXCM 支持订阅实时流。`fxcmpy` 通过回调函数（callback）异步处理到达的数据：


In [ ]:
def output(data, dataframe):
    """简单回调：打印条数、品种、时间、买一/卖一。"""
    print(
        "%3d | %s | %s | %6.5f, %6.5f"
        % (
            len(dataframe),
            data["Symbol"],
            pd.to_datetime(int(data["Updated"]), unit="ms"),
            data["Rates"][0],
            data["Rates"][1],
        )
    )


# api.subscribe_market_data("EUR/USD", (output,))   # ① 订阅；可挂多个回调
# api.get_last_price("EUR/USD")                     # ② 订阅期间取最新价
# api.unsubscribe_market_data("EUR/USD")            # ③ 取消订阅
print("流式订阅：取消注释后在交易时段运行；用完务必 unsubscribe。")


回调可只做打印，也可在其中计算信号并触发下单，是实时策略的常见挂载点。


## 4. Placing Orders

FXCM 支持交易端可见的各类订单（入场单、追踪止损等）。入门阶段，市价买卖通常足够。

演示账户默认常为 hedge 账户：同品种做多 10,000 再做空 10,000 会形成两笔持仓，而不是像 Oanda 默认净持仓那样轧差。数量单位对货币对通常以千为单位（`amountK`）；杠杆与保证金因账户而异，请按账户设置缩小示例手数。


In [ ]:
sel = ["tradeId", "amountK", "currency", "grossPL", "isBuy"]

# api.get_open_positions()                          # ① 当前持仓（空则无行）
# order = api.create_market_buy_order("EUR/USD", 100)   # ② 买入 100×1000 = 100,000 EUR/USD
# api.get_open_positions()[sel]
# order = api.create_market_buy_order("EUR/GBP", 50)    # ③ 另一品种
# api.get_open_positions()[sel]

# order = api.create_market_sell_order("EUR/USD", 25)   # ④ 卖出，减小/反转仓位
# order = api.create_market_buy_order("EUR/GBP", 50)    # ⑤ 再增 EUR/GBP（hedge 下可能多笔）
# api.get_open_positions()[sel]

# api.close_all_for_symbol("EUR/GBP")                   # ⑥ 平掉该品种全部持仓
# api.close_all()                                       # ⑦ 平掉所有持仓
# api.get_open_positions()
print("下单示例已写在注释中。无实盘/Demo 连接时请勿取消注释盲跑。")


## 5. Account Information


In [ ]:
# api.get_default_account()   # 默认 accountId（多账户时）
# api.get_accounts().T        # 权益、可用保证金、是否 hedging 等

# 书中示意字段：accountId, balance, equity, usableMargin, hedging, dayPL, grossPL ...
print("账户查询：连接成功后运行 get_default_account / get_accounts。")


## 6. 与第 8 章 Oanda 流水线对照

| 步骤 | Oanda（`tpqoa`） | FXCM（`fxcmpy`） | 天勤（本仓库） |
|---|---|---|---|
| 配置 | `oanda.cfg` | `fxcm.cfg` | `tqsdk.cfg` |
| 连接 | `tpqoa.tpqoa(...)` | `fxcmpy.fxcmpy(...)` | `TqApi(auth=TqAuth(...), account=...)` |
| 品种 | `get_instruments()` | `get_instruments()` | `query_quotes` / `get_quote` |
| 历史 | `get_history(...)` | `get_candles(...)` / tick·candles reader | `get_kline_serial` |
| 流式 | `stream_data(...)` | `subscribe_market_data` + 回调 | `wait_update` + `is_changing` |
| 下单 | `create_order(...)` | `create_market_buy/sell_order` | `TargetPosTask.set_target_volume` |
| 账户 | `get_account_summary` | `get_accounts` / `get_open_positions` | `get_account` / `get_position` / `get_trade` |

三者都是同一条算法交易流水线；差异在标的、账户模型与辖区，而非「要不要写策略代码」本身。


## 7. Conclusions

本章覆盖 FXCM REST API 与 `fxcmpy` 的基础能力：

1. 安装与 Demo/实盘连接  
2. 预打包 tick / candles 下载  
3. API 历史 candles  
4. 流式行情与回调  
5. 市价开平与按品种/全部清仓  
6. 账户与持仓查询  

更细的订单类型见 http://fxcmpy.tpq.io。Oanda 与 FXCM 为境外两套通道；国内可复现的期货算法交易链路以天勤笔记为准。


## 8. References

- FXCM Trading API：https://fxcm.github.io/rest-api-docs  
- fxcmpy：http://fxcmpy.tpq.io  
- 原书：Yves Hilpisch, *Python for Algorithmic Trading*, Chapter 9  
- 本仓库对照：`CFD_Trading_with_Oanda.ipynb`、`天勤TqSdk期货算法交易.ipynb`
